### Pandas时间序列

In [2]:
import numpy as np
import pandas as pd

#### 时间戳

In [3]:
# 1.创建
pd.Timestamp('2023-01-01')
pd.Period('2023-01-01',freq='D')
# freq:Y:年 M:月 D:日

# 批量生成时刻数据
# periods=4 创建4个时间
index = pd.date_range('2030.02.13',periods=4,freq='D') # 时刻数据
# index = pd.period_range('2030.02.13',periods=4,freq='D') # 时期数据
index

# 时间戳索引
pd.Series(np.random.randint(0, 10, size=4),index=index)

2030-02-13    3
2030-02-14    2
2030-02-15    6
2030-02-16    8
Freq: D, dtype: int32

In [4]:
# 2.转换方法
# pd.to_datetime(['2030.03.14', '2030-03-14', '14/03/2030', '2030/3/14'])

# 时间戳->时间
pd.to_datetime([1899678987], unit='s')

dt = pd.to_datetime([1899678987], unit='ms')
display(dt)

# 时间差 DateOffset
dt + pd.DateOffset(hours=8) # +8小时
dt + pd.DateOffset(days=8) # +8天
dt + pd.DateOffset(days=-8) #-8天

DatetimeIndex(['1970-01-22 23:41:18.987000'], dtype='datetime64[ns]', freq=None)

DatetimeIndex(['1970-01-14 23:41:18.987000'], dtype='datetime64[ns]', freq=None)

In [5]:
# 3.时间戳的索引和切片
index = pd.date_range('2030-3-14', periods=100, freq='D')
index

ts = pd.Series(range(len(index)), index=index)
ts

2030-03-14     0
2030-03-15     1
2030-03-16     2
2030-03-17     3
2030-03-18     4
              ..
2030-06-17    95
2030-06-18    96
2030-06-19    97
2030-06-20    98
2030-06-21    99
Freq: D, Length: 100, dtype: int64

In [6]:
# 索引
ts['2030-03-15'] # 索引
ts['2030-03'] # 3月份
ts['2030'] # 2030年

ts['2030-03-15':'2030-03-20'] # 2030年3月15日到20日

2030-03-15    1
2030-03-16    2
2030-03-17    3
2030-03-18    4
2030-03-19    5
2030-03-20    6
Freq: D, dtype: int64

In [7]:
# 时间戳索引
pd.Timestamp('2030-3-22')
ts[pd.Timestamp('2030-3-22')]

# 切片
ts[pd.Timestamp('2030-3-15'):pd.Timestamp('2030-3-22')]

# date_range
ts[pd.date_range('2030-3-24',periods=10, freq='D')]

2030-03-24    10
2030-03-25    11
2030-03-26    12
2030-03-27    13
2030-03-28    14
2030-03-29    15
2030-03-30    16
2030-03-31    17
2030-04-01    18
2030-04-02    19
Freq: D, dtype: int64

In [8]:
# 4.属性
ts.index

ts.index.year
ts.index.month
ts.index.day
ts.index.dayofweek

Index([3, 4, 5, 6, 0, 1, 2, 3, 4, 5, 6, 0, 1, 2, 3, 4, 5, 6, 0, 1, 2, 3, 4, 5,
       6, 0, 1, 2, 3, 4, 5, 6, 0, 1, 2, 3, 4, 5, 6, 0, 1, 2, 3, 4, 5, 6, 0, 1,
       2, 3, 4, 5, 6, 0, 1, 2, 3, 4, 5, 6, 0, 1, 2, 3, 4, 5, 6, 0, 1, 2, 3, 4,
       5, 6, 0, 1, 2, 3, 4, 5, 6, 0, 1, 2, 3, 4, 5, 6, 0, 1, 2, 3, 4, 5, 6, 0,
       1, 2, 3, 4],
      dtype='int32')

#### 时间序列常用方法
- 对时间做一些移动/滞后、频率转换、采样等操作

In [11]:
index = pd.date_range('2030-3-1', periods=365, freq='D')
ts = pd.Series(np.random.randint(0, 500, len(index)), index=index)
ts

2030-03-01     23
2030-03-02    404
2030-03-03    362
2030-03-04    365
2030-03-05    405
             ... 
2031-02-24    267
2031-02-25    440
2031-02-26     12
2031-02-27    170
2031-02-28    189
Freq: D, Length: 365, dtype: int32

In [14]:
# 1.移动
ts.shift() # 向后移动1天
ts.shift(periods=2) # 向后移动2位

2030-03-01      NaN
2030-03-02      NaN
2030-03-03     23.0
2030-03-04    404.0
2030-03-05    362.0
              ...  
2031-02-24    320.0
2031-02-25    347.0
2031-02-26    267.0
2031-02-27    440.0
2031-02-28     12.0
Freq: D, Length: 365, dtype: float64

In [20]:
# 2.频率转换
ts.asfreq(pd.tseries.offsets.Week()) # 天 -> 周
ts.asfreq(pd.tseries.offsets.MonthEnd()) # 天 -> 月

# 由少变多 fill_value 填充
ts.asfreq(pd.tseries.offsets.Hour(), fill_value=0) # 天 -> 小时

2030-03-01 00:00:00     23
2030-03-01 01:00:00      0
2030-03-01 02:00:00      0
2030-03-01 03:00:00      0
2030-03-01 04:00:00      0
                      ... 
2031-02-27 20:00:00      0
2031-02-27 21:00:00      0
2031-02-27 22:00:00      0
2031-02-27 23:00:00      0
2031-02-28 00:00:00    189
Freq: h, Length: 8737, dtype: int32

#### resample:根据日期维度进行数据聚合
- 按照分钟(T)、小时(H)、日(D)、周(W)、月(M)、年(Y)等作为日期维度

In [21]:
ts

2030-03-01     23
2030-03-02    404
2030-03-03    362
2030-03-04    365
2030-03-05    405
             ... 
2031-02-24    267
2031-02-25    440
2031-02-26     12
2031-02-27    170
2031-02-28    189
Freq: D, Length: 365, dtype: int32

In [33]:
# 3.重采样
ts.resample('2D').sum() # 按天聚合求和
ts.resample('2W').sum() # 按2个星期为单位进行汇总，求和
ts.resample('3ME').sum() # 按3个月为单位进行汇总，求和
ts.resample('3ME').sum().cumsum() # 按3个月为单位进行汇总，求和，并累加

ts.resample('h').sum()
ts.resample('min').sum()
ts.resample('s').sum()

2030-03-01 00:00:00     23
2030-03-01 00:00:01      0
2030-03-01 00:00:02      0
2030-03-01 00:00:03      0
2030-03-01 00:00:04      0
                      ... 
2031-02-27 23:59:56      0
2031-02-27 23:59:57      0
2031-02-27 23:59:58      0
2031-02-27 23:59:59      0
2031-02-28 00:00:00    189
Freq: s, Length: 31449601, dtype: int32

In [34]:
# 4.DataFrame重采样
d = {
    'price': [10, 11, 2, 44, 33, 44, 55, 66],
    'score': [40, 30, 20, 50, 60, 70, 80, 10],
    'week': pd.date_range('2030-3-1', periods=8, freq='W')
}

df = pd.DataFrame(d)
df

,price,score,week
0,10,40,2030-03-03
1,11,30,2030-03-10
2,2,20,2030-03-17
3,44,50,2030-03-24
4,33,60,2030-03-31
5,44,70,2030-04-07
6,55,80,2030-04-14
7,66,10,2030-04-21


In [43]:
# 对week列进行按月汇总求和
df.resample('ME', on='week').sum()
df.resample('ME', on='week').apply(lambda x: x.sum())

# 对week列进行按月汇总求和 ，并计算平均价格 和总分数
df.resample('ME', on='week').agg({'price': 'mean', 'score': 'sum'})

,price,score
week,,
2030-03-31,20.0,200
2030-04-30,55.0,160


#### 时区

In [51]:
index = pd.date_range('2030-3-1', periods=3, freq='D')
ts = pd.Series(np.random.randn(len(index)), index=index)
ts

2030-03-01   -1.565496
2030-03-02   -0.526696
2030-03-03    1.001550
Freq: D, dtype: float64

In [47]:
# tz: timezone
import pytz

In [48]:
pytz.common_timezones

['Africa/Abidjan', 'Africa/Accra', 'Africa/Addis_Ababa', 'Africa/Algiers', 'Africa/Asmara', 'Africa/Bamako', 'Africa/Bangui', 'Africa/Banjul', 'Africa/Bissau', 'Africa/Blantyre', 'Africa/Brazzaville', 'Africa/Bujumbura', 'Africa/Cairo', 'Africa/Casablanca', 'Africa/Ceuta', 'Africa/Conakry', 'Africa/Dakar', 'Africa/Dar_es_Salaam', 'Africa/Djibouti', 'Africa/Douala', 'Africa/El_Aaiun', 'Africa/Freetown', 'Africa/Gaborone', 'Africa/Harare', 'Africa/Johannesburg', 'Africa/Juba', 'Africa/Kampala', 'Africa/Khartoum', 'Africa/Kigali', 'Africa/Kinshasa', 'Africa/Lagos', 'Africa/Libreville', 'Africa/Lome', 'Africa/Luanda', 'Africa/Lubumbashi', 'Africa/Lusaka', 'Africa/Malabo', 'Africa/Maputo', 'Africa/Maseru', 'Africa/Mbabane', 'Africa/Mogadishu', 'Africa/Monrovia', 'Africa/Nairobi', 'Africa/Ndjamena', 'Africa/Niamey', 'Africa/Nouakchott', 'Africa/Ouagadougou', 'Africa/Porto-Novo', 'Africa/Sao_Tome', 'Africa/Tripoli', 'Africa/Tunis', 'Africa/Windhoek', 'America/Adak', 'America/Anchorage', 'Amer

In [52]:
# 时区表式
ts = ts.tz_localize(tz = 'UTC')
ts

2030-03-01 00:00:00+00:00   -1.565496
2030-03-02 00:00:00+00:00   -0.526696
2030-03-03 00:00:00+00:00    1.001550
Freq: D, dtype: float64

In [53]:
# 时区转换
ts.tz_convert('Asia/Shanghai')

2030-03-01 08:00:00+08:00   -1.565496
2030-03-02 08:00:00+08:00   -0.526696
2030-03-03 08:00:00+08:00    1.001550
Freq: D, dtype: float64